In [1]:
"""
Sample-adequacy analysis for CB-SEM (MacCallum, Browne & Sugawara, 1996).
Computes model df, free parameters, RMSEA-based power and minimum N.
Requires: scipy
"""
from scipy.stats import ncx2, ncf, f as f_dist
from scipy.optimize import brentq

# ---------------- 1. CORE FUNCTIONS ----------------
def sem_df(items, exog, endo, paths):
    """
    items: {construct: number_of_indicators}
    exog : exogenous latent constructs (variances + covariances free)
    endo : endogenous latent constructs (one disturbance variance each)
    paths: list of (source, target) structural paths
    Assumes marker-variable scaling, no cross-loadings, no correlated errors,
    uncorrelated disturbances, all exogenous latents correlated.
    """
    for s, t in paths:
        assert s in items and t in items, f"Unknown construct in path {s}->{t}"
        assert t in endo, f"Path target {t} must be endogenous"
    k = len(items)
    p = sum(items.values())
    moments = p * (p + 1) // 2
    measurement = (p - k) + p                 # free loadings + error variances
    ne = len(exog)
    latent = ne * (ne + 1) // 2 + len(endo) + len(paths)
    npar = measurement + latent
    cfa_npar = measurement + k * (k + 1) // 2
    return {"indicators": p, "constructs": k, "moments": moments,
            "npar": npar, "df": moments - npar,
            "cfa_npar": cfa_npar, "cfa_df": moments - cfa_npar}

def rmsea_power(df, n, eps0=0.05, epsa=0.08, alpha=0.05):
    """epsa > eps0: test of close fit; epsa < eps0: test of not-close fit."""
    l0 = (n - 1) * df * eps0 ** 2
    la = (n - 1) * df * epsa ** 2
    if epsa > eps0:
        crit = ncx2.ppf(1 - alpha, df, l0)
        return 1 - ncx2.cdf(crit, df, la)
    crit = ncx2.ppf(alpha, df, l0)
    return ncx2.cdf(crit, df, la)

def min_n(df, target=0.80, **kw):
    g = lambda n: rmsea_power(df, n, **kw) - target
    lo, hi = 5, 200000
    return lo if g(lo) >= 0 else brentq(g, lo, hi)

def reg_min_n(u, f2=0.15, alpha=0.05, power=0.90):
    """Regression-based minimum N (fixed-model R-squared test, as in G*Power)."""
    def pw(n):
        d2 = n - u - 1
        crit = f_dist.ppf(1 - alpha, u, d2)
        return 1 - ncf.cdf(crit, u, d2, f2 * n) - power
    return int(brentq(pw, u + 3, 10000)) + 1

# ---------------- 2. MODEL DEFINITIONS ----------------
MODELS = {}


# Manuscript 5: WR->RI, DIC->DIQ, SP->SA -> RCC -> SCR ; EU moderates RCC->SCR
MODELS["M5"] = dict(
    n=405,
    items={"WR": 6, "RI": 6, "DIC": 6, "DIQ": 6, "SP": 6, "SA": 6,
           "RCC": 6, "SCR": 6, "EU": 6},
    exog=["WR", "DIC", "SP", "EU"], endo=["RI", "DIQ", "SA", "RCC", "SCR"],
    paths=[("WR", "RI"), ("DIC", "DIQ"), ("SP", "SA"), ("RI", "RCC"), ("DIQ", "RCC"),
           ("SA", "RCC"), ("RCC", "SCR"), ("EU", "SCR")],
    direct=[("WR", "RCC"), ("DIC", "RCC"), ("SP", "RCC")],
    max_predictors=3,
)

# ---------------- 3. RUN ----------------
def run(m, alpha=0.05):
    out = []
    for label, extra in (("Hypothesised paths", []), ("+ direct paths", m["direct"])):
        r = sem_df(m["items"], m["exog"], m["endo"], m["paths"] + extra)
        n = m["n"]
        r.update(label=label,
                 power_close=rmsea_power(r["df"], n, 0.05, 0.08, alpha),
                 power_notclose=rmsea_power(r["df"], n, 0.05, 0.01, alpha),
                 minN_close=min_n(r["df"], 0.80, eps0=0.05, epsa=0.08, alpha=alpha),
                 ratio=n / r["npar"])
        out.append(r)
    return out

if __name__ == "__main__":
    for name, m in MODELS.items():
        print(f"\n=== {name}  (N = {m['n']}) ===")
        for r in run(m):
            print(f"{r['label']:<20} indicators={r['indicators']:>3}  npar={r['npar']:>4}  "
                  f"df={r['df']:>5}  CFA df={r['cfa_df']:>5}  "
                  f"power(close)={r['power_close']:.3f}  "
                  f"power(not close)={r['power_notclose']:.3f}  "
                  f"minN(.80)={r['minN_close']:.0f}  N:npar={r['ratio']:.2f}")
        print("Regression-based check (f2=.15, power=.90, alpha=.05): "
              f"min N for {m['max_predictors']} predictors = {reg_min_n(m['max_predictors'])}")


=== M5  (N = 405) ===
Hypothesised paths   indicators= 54  npar= 122  df= 1363  CFA df= 1341  power(close)=1.000  power(not close)=1.000  minN(.80)=28  N:npar=3.32
+ direct paths       indicators= 54  npar= 125  df= 1360  CFA df= 1341  power(close)=1.000  power(not close)=1.000  minN(.80)=28  N:npar=3.24
Regression-based check (f2=.15, power=.90, alpha=.05): min N for 3 predictors = 99
